<a href="https://colab.research.google.com/github/anshikait/M.A.R.G/blob/main/M.A.R.G(testcases).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

GPU: True Tesla T4


In [2]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
shutil.rmtree("/content/M.A.R.G", ignore_errors=True)
!git clone https://github.com/anshikait/M.A.R.G.git /content/M.A.R.G
%cd /content/M.A.R.G
!pip install -q -r requirements.txt
os.makedirs("/content/models", exist_ok=True)

def check(p):
    f = os.path.join(p, "model.safetensors")
    print(p, "->", f"{os.path.getsize(f)/1e9:.2f} GB" if os.path.exists(f) else "MISSING")

Mounted at /content/drive
Cloning into '/content/M.A.R.G'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 119 (delta 34), reused 104 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 22.86 MiB | 23.24 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/M.A.R.G
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [25]:
%%writefile /content/M.A.R.G/eval/evaluate_rerank.py
"""Tuned dense retriever -> top-POOL -> cross-encoder rerank. Prints dense vs reranked side by side. (Self-contained.)"""
import argparse
import numpy as np, pandas as pd
from configs import settings as S
from src.data_prep import loaders as L
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker, rerank_orders


def metrics(orders, slugs, q):
    pos = {s: i for i, s in enumerate(slugs)}
    ranks, anyrel, prel = [], [], []
    for gold, rel, order in zip(q["slug"], q["relevant"], orders):
        order = list(order)
        ranks.append(order.index(pos[gold]) + 1 if pos.get(gold) in order else 10**6)
        top3 = {slugs[i] for i in order[:3]}; rel = set(rel) | {gold}
        anyrel.append(bool(top3 & rel)); prel.append(len(top3 & rel) / 3)
    r = np.array(ranks)
    return pd.DataFrame({"hit3": r <= 3, "hit10": r <= 10, "hit20": r <= 20, "hit30": r <= 30,
                         "rr": np.where(r < 10**6, 1 / r, 0.0), "anyrel3": anyrel, "prel3": prel}, index=q.index)


def summary(m):
    return (f"Recall@3={m.hit3.mean():.3f}  Recall@10={m.hit10.mean():.3f}  Recall@20={m.hit20.mean():.3f}  "
            f"Recall@30={m.hit30.mean():.3f}  MRR={m.rr.mean():.3f}  | any-relevant@3={m.anyrel3.mean():.3f}  "
            f"relevant-precision@3={m.prel3.mean():.3f}")


def table(q, m, by):
    g = pd.concat([q[[by]], m], axis=1).groupby(by)
    t = g.agg(n=("hit3", "size"), R3=("hit3", "mean"), R10=("hit10", "mean"), R30=("hit30", "mean"),
              MRR=("rr", "mean"), anyrel3=("anyrel3", "mean"))
    return t.sort_values("n", ascending=False).round(3)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="val"); ap.add_argument("--index", choices=["all", "split"], default="all")
    ap.add_argument("--retriever", required=True); ap.add_argument("--reranker", required=True)
    ap.add_argument("--pool", type=int, default=30); ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--include-vague", action="store_true")
    a = ap.parse_args()
    docs = L.card_frame()
    if a.index == "split": docs = docs[docs.split == a.split]
    q = L.load_queries(a.split); q = q[q.slug.notna() & q.slug.isin(docs.slug)].copy()
    if not a.include_vague: q = q[q.row_type != "vague"].copy()
    if a.limit and a.limit < len(q): q = q.sample(a.limit, random_state=0).copy()
    print(f"index={a.index} docs={len(docs)} | scored queries={len(q)} | pool={a.pool}", flush=True)
    ret = HybridRetriever(docs, use_dense=True, model_name=a.retriever)
    orders = [[i for i, _ in lst] for lst in ret.rank_lists(q["query"].tolist(), "dense", pool=max(100, a.pool))]
    slugs, texts = ret.df["slug"].tolist(), ret.df["text"].tolist()
    base = metrics(orders, slugs, q)
    print("\n######## dense (no rerank)  ", summary(base), flush=True)
    rr = CrossEncoderReranker(a.reranker)
    new = metrics(rerank_orders(orders, texts, q["query"].tolist(), rr.score, a.pool), slugs, q)
    print("######## dense + reranker   ", summary(new))
    S.RESULTS.mkdir(parents=True, exist_ok=True)
    for by in ("language", "noise", "row_type"):
        t = table(q, new, by); print(f"\n-- reranked, by {by}\n{t.to_string()}")
        t.to_csv(S.RESULTS / f"rerank_{a.split}_{a.index}_pool{a.pool}_by_{by}.csv")

Writing /content/M.A.R.G/eval/evaluate_rerank.py


In [4]:
%cd /content/M.A.R.G
!ls eval src/retrieval
!echo "score_pairs:";  grep -c "def score_pairs" src/retrieval/reranker.py
!echo "rerank_orders:"; grep -c "def rerank_orders" src/retrieval/reranker.py
!echo "freeze-layers:"; grep -c "freeze-layers" src/retrieval/finetune_retriever.py
!echo "bm25_weight:";  grep -c "bm25_weight" src/retrieval/hybrid_retriever.py
!echo "hit30:";        grep -c "hit30" eval/evaluate_retrieval.py

/content/M.A.R.G
eval:
evaluate_constraints.py  evaluate_nlu.py     evaluate_retrieval.py  __pycache__
evaluate_eligibility.py  evaluate_rerank.py  __init__.py	    results

src/retrieval:
finetune_reranker.py   hybrid_retriever.py  __pycache__
finetune_retriever.py  __init__.py	    reranker.py
score_pairs:
1
rerank_orders:
1
freeze-layers:
2
bm25_weight:
4
hit30:
4


In [5]:
!python -m src.data_prep.check_data | tail -25

  "registry_not_in_split": 0,
  "cards_not_in_registry": 0,
  "query_leaks": {
    "train": {
      "queries": 8407,
      "target_in_wrong_split": 0,
      "target_not_in_split_map": 0,
      "no_target(offtopic)": 163
    },
    "val": {
      "queries": 1795,
      "target_in_wrong_split": 0,
      "target_not_in_split_map": 0,
      "no_target(offtopic)": 32
    },
    "test": {
      "queries": 1798,
      "target_in_wrong_split": 0,
      "target_not_in_split_map": 0,
      "no_target(offtopic)": 33
    }
  }
}

PASS


In [6]:
!python -m eval.evaluate_retrieval --split val --modes bm25 --index split

index=split docs=746 | scored queries=1610 | state_filter=none
doc state values: {'All': 109, 'Gujarat': 90, 'Uttarakhand': 54, 'Rajasthan': 49} | levels: {'state': 634, 'central': 112}

######## bm25   Recall@3=0.335  Recall@10=0.543  Recall@20=0.685  Recall@30=0.760  MRR=0.304  | any-relevant@3=0.400  relevant-precision@3=0.148

-- by language
              n     R3    R10    R30    MRR  anyrel3
language                                            
english    1095  0.376  0.601  0.818  0.330    0.427
tamil        43  0.326  0.558  0.721  0.313    0.395
malayalam    43  0.302  0.465  0.674  0.310    0.488
urdu         42  0.238  0.452  0.643  0.279    0.286
odia         40  0.275  0.550  0.675  0.289    0.325
assamese     40  0.225  0.450  0.700  0.236    0.350
kannada      40  0.225  0.350  0.550  0.221    0.275
hindi        37  0.270  0.405  0.649  0.228    0.459
punjabi      35  0.314  0.429  0.600  0.246    0.400
hinglish     35  0.057  0.171  0.371  0.080    0.057
bengali      34 

In [7]:
!python -m eval.evaluate_retrieval --split val --modes bm25 dense hybrid --index all

index=all docs=4972 | scored queries=1610 | state_filter=none
doc state values: {'All': 705, 'Gujarat': 616, 'Uttarakhand': 411, 'Rajasthan': 327} | levels: {'state': 4242, 'central': 730}
modules.json: 100% 349/349 [00:00<00:00, 1.85MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 789kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 26.5MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 262kB/s]
config.json: 100% 687/687 [00:00<00:00, 4.57MB/s]

pytorch_model.bin: downloading bytes:  30% 672M/2.27G [00:02<00:04, 333MB/s, 55.3MB/s  ]
pytorch_model.bin: downloading bytes:  37% 835M/2.27G [00:03<00:05, 279MB/s, 68.5MB/s  ]
pytorch_model.bin: downloading bytes:  56% 1.27G/2.27G [00:17<00:05, 194MB/s, 51.1MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:28<00:12, 61.5MB/s, 38.7MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:48<00:12, 61.5MB/s, 38.8MB/s  ]
pytorch_model.bin: reconstructing file:  86% 1.94G/2.27G [00:50<00:09, 

In [8]:
!python -m src.retrieval.finetune_retriever --freeze-layers 8 --epochs 1 --out /content/models/bge_m3_janora

train queries=7591 | train cards=3481 | val queries=1610 | device=cuda
Loading weights: 100% 391/391 [00:00<00:00, 27819.73it/s]
base model on val (split index): {'R@3': 0.4763975155279503, 'R@10': 0.6161490683229813}
gradient checkpointing: on
frozen lower layers: 8
trainable parameters: 203M of 568M
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/content/M.A.R.G/src/retrieval/finetune_retriever.py:141: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); step += 1; run += loss.item()
  ep1 step 50/475 loss=1.3137
  ep1 step 100/475 loss=0.4908
  e

In [9]:
check("/content/models/bge_m3_janora")          # must print about 2.2 GB
shutil.copytree("/content/models/bge_m3_janora", "/content/drive/MyDrive/janora_models/bge_m3_janora_v2", dirs_exist_ok=True)
drive.flush_and_unmount(); drive.mount("/content/drive")
check("/content/drive/MyDrive/janora_models/bge_m3_janora_v2")   # must also print about 2.2 GB

/content/models/bge_m3_janora -> 2.27 GB
Mounted at /content/drive
/content/drive/MyDrive/janora_models/bge_m3_janora_v2 -> 2.27 GB


In [10]:
!python -m eval.evaluate_retrieval --split val --modes dense --index all --model /content/models/bge_m3_janora

index=all docs=4972 | scored queries=1610 | state_filter=none
doc state values: {'All': 705, 'Gujarat': 616, 'Uttarakhand': 411, 'Rajasthan': 327} | levels: {'state': 4242, 'central': 730}
Loading weights: 100% 391/391 [00:00<00:00, 17406.34it/s]
Batches: 100% 156/156 [02:47<00:00,  1.07s/it]

######## dense   Recall@3=0.642  Recall@10=0.789  Recall@20=0.864  Recall@30=0.917  MRR=0.580  | any-relevant@3=0.857  relevant-precision@3=0.521

-- by language
              n     R3    R10    R30    MRR  anyrel3
language                                            
english    1095  0.761  0.889  0.972  0.692    0.899
tamil        43  0.465  0.651  0.884  0.440    0.884
malayalam    43  0.395  0.605  0.814  0.406    0.814
urdu         42  0.429  0.571  0.714  0.333    0.810
odia         40  0.375  0.650  0.875  0.319    0.700
assamese     40  0.350  0.500  0.775  0.308    0.675
kannada      40  0.350  0.550  0.750  0.299    0.800
hindi        37  0.459  0.703  0.811  0.410    0.784
punjabi      

In [11]:
!python -m src.retrieval.finetune_reranker --retriever /content/models/bge_m3_janora --out /content/models/bge_reranker_janora --freeze-layers 8

train queries=7591 | val subset=300 | device=cuda
Loading weights: 100% 391/391 [00:00<00:00, 15595.77it/s]
retriever-only R@3 on this val subset (val-split index): 0.873  | pool recall: 1.000
config.json: 100% 795/795 [00:00<00:00, 3.65MB/s]
tokenizer_config.json: 100% 1.17k/1.17k [00:00<00:00, 4.34MB/s]

tokenizer.json: downloading bytes:   6% 967k/17.1M [00:00<00:10, 1.58MB/s]
tokenizer.json: downloading bytes: 100% 5.88M/5.88M [00:00<00:00, 7.77MB/s,  578kB/s  ]
tokenizer.json: reconstructing file: 100% 17.1M/17.1M [00:00<00:00, 22.6MB/s, 1.68MB/s  ]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 4.78MB/s]

model.safetensors: downloading bytes:   7% 169M/2.27G [00:01<00:09, 231MB/s, 13.6MB/s  ]
model.safetensors: downloading bytes:  17% 383M/2.27G [00:02<00:10, 174MB/s, 32.4MB/s  ]
model.safetensors: downloading bytes:  37% 851M/2.27G [00:04<00:08, 176MB/s, 64.1MB/s  ]
model.safetensors: downloading bytes:  39% 896M/2.27G [00:14<01:49, 12.5MB/s, 31.7MB/s  ]
model.safetensors: 

In [12]:
check("/content/models/bge_reranker_janora")
shutil.copytree("/content/models/bge_reranker_janora", "/content/drive/MyDrive/janora_models/bge_reranker_janora_v2", dirs_exist_ok=True)
drive.flush_and_unmount(); drive.mount("/content/drive")
check("/content/drive/MyDrive/janora_models/bge_reranker_janora_v2")

/content/models/bge_reranker_janora -> 2.27 GB
Mounted at /content/drive
/content/drive/MyDrive/janora_models/bge_reranker_janora_v2 -> 2.27 GB


In [13]:
!python -m eval.evaluate_rerank --split val --index all --retriever /content/models/bge_m3_janora --reranker /content/models/bge_reranker_janora

index=all docs=4972 | scored queries=1610 | pool=30
Loading weights: 100% 391/391 [00:00<00:00, 720.23it/s]

######## dense (no rerank)   Recall@3=0.642  Recall@10=0.789  Recall@20=0.864  Recall@30=0.917  MRR=0.580  | any-relevant@3=0.857  relevant-precision@3=0.521
Loading weights: 100% 393/393 [00:00<00:00, 13481.32it/s]
######## dense + reranker    Recall@3=0.694  Recall@10=0.829  Recall@20=0.893  Recall@30=0.917  MRR=0.636  | any-relevant@3=0.934  relevant-precision@3=0.613

-- reranked, by language
              n     R3    R10    R30    MRR  anyrel3
language                                            
english    1095  0.814  0.917  0.972  0.745    0.960
tamil        43  0.535  0.698  0.884  0.491    0.953
malayalam    43  0.512  0.651  0.814  0.435    0.860
urdu         42  0.548  0.619  0.714  0.504    0.952
odia         40  0.475  0.700  0.875  0.447    0.875
assamese     40  0.300  0.575  0.775  0.285    0.775
kannada      40  0.300  0.575  0.750  0.281    0.875
hindi        3

In [14]:
!python -m eval.evaluate_nlu --split val
!python -m eval.evaluate_eligibility --split val

INTENT (rules)  acc=0.344  macroF1=0.472
INTENT (tfidf+LR) acc=0.985  macroF1=0.972
              precision    recall  f1-score   support

       named       0.94      0.87      0.91       132
    offtopic       1.00      1.00      1.00        32
    standard       0.99      1.00      0.99      1478
       vague       1.00      0.98      0.99       153

    accuracy                           0.98      1795
   macro avg       0.98      0.96      0.97      1795
weighted avg       0.98      0.98      0.98      1795


SLOTS (English rows, regex baseline)
          attr   tp  fp  fn     P     R    F1
           age  886  29 227 0.968 0.796 0.874
        gender  655   1 426 0.998 0.606 0.754
         state 1008   9 182 0.991 0.847 0.913
caste_category  464   7 520 0.985 0.472 0.638
 annual_income  382  47 617 0.890 0.382 0.535
    occupation  514   4 644 0.992 0.444 0.613
    disability   60   0  35 1.000 0.632 0.774
           bpl  222   0  19 1.000 0.921 0.959
     residence  386   1 684 0

In [15]:
import sys; sys.path.insert(0, "/content/M.A.R.G")
from src.data_prep import loaders as L
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker
docs = L.card_frame()
ret = HybridRetriever(docs, use_dense=True, model_name="/content/models/bge_m3_janora")
rr  = CrossEncoderReranker("/content/models/bge_reranker_janora")

def ask(q, k=3, pool=30):
    hits = ret.search(q, k=pool, mode="dense")
    order, _ = rr.rerank(q, hits["text"].tolist())
    print(f"\nQ: {q}\n  retriever:", [r.name for r in hits.head(k).itertuples()])
    print("  reranked: ", [r.name for r in hits.iloc[order[:k]].itertuples()])

ask("I am a 45 year old farmer from Gujarat, income 80000, which scheme can help me?")
ask("mujhe scholarship chahiye main SC student hu")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Q: I am a 45 year old farmer from Gujarat, income 80000, which scheme can help me?
  retriever: ['National Mission for Sustainable Agriculture - Reinforced Area Development', 'Harvesting Tools Assistance for General Category (HRT-2)', 'National Mission on Edible Oil- Oil Palm: Drip Irrigation']
  reranked:  ['Wire Fencing Around the Farm for Crop Protection', 'Assistance for Area Expansion in Fruit crops', 'To Assist the Farmers for Construction of Community Base Water Tank/Storage Structure for Efficient Use of Water Through Drip Irrigation']

Q: mujhe scholarship chahiye main SC student hu
  retriever: ['Top Class Education For Scheduled Caste Students', 'Post-Graduate Scholarships for Professional Courses for SC/ST Candidates', 'Solar Energy Corporation of India (SECI) Internship']
  reranked:  ['Top Class Education For Scheduled Caste Students', 'Post-Graduate Scholarships for Professional Courses for SC/ST Candidates', 'Post Matric Scholarship for SC students (Delhi)']


In [16]:
!mkdir -p /content/drive/MyDrive/janora_results
!cp -r eval/results/* /content/drive/MyDrive/janora_results/ 2>/dev/null; ls /content/drive/MyDrive/janora_results | tail -5

retrieval_val_split_bm25_by_noise.csv
retrieval_val_split_bm25_by_row_type.csv
retrieval_val_split_none_bm25_by_language.csv
retrieval_val_split_none_bm25_by_noise.csv
retrieval_val_split_none_bm25_by_row_type.csv


In [19]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/anshikait/M.A.R.G.git
%cd /content/M.A.R.G
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'M.A.R.G' already exists and is not an empty directory.
/content/M.A.R.G


In [20]:
import os, shutil

os.makedirs('/content/models', exist_ok=True)

shutil.copytree(
    '/content/drive/MyDrive/janora_models/bge_m3_janora_v2',
    '/content/models/bge_m3_janora',
    dirs_exist_ok=True
)

shutil.copytree(
    '/content/drive/MyDrive/janora_models/bge_reranker_janora_v2',
    '/content/models/bge_reranker_janora',
    dirs_exist_ok=True
)

'/content/models/bge_reranker_janora'

In [21]:
import os

print(os.path.exists("/content/models/bge_m3_janora"))
print(os.path.exists("/content/drive/MyDrive/janora_models/bge_m3_janora_v2"))

True
True


In [22]:
import os

print("Repository exists:", os.path.exists("/content/M.A.R.G"))

if os.path.exists("/content/M.A.R.G"):
    print("\nFiles in repository:")
    print(os.listdir("/content/M.A.R.G"))

    print("\nDoes src exist?",
          os.path.exists("/content/M.A.R.G/src"))

Repository exists: True

Files in repository:
['README.md', 'data', '.git', 'M.A.R.G.ipynb', 'src', 'requirements.txt', 'RerankerDone.ipynb', 'eval', 'janora_colab.zip', 'configs']

Does src exist? True


In [24]:
import os, shutil, sys
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists("/content/M.A.R.G"):
    !git clone https://github.com/anshikait/M.A.R.G.git /content/M.A.R.G
%cd /content/M.A.R.G
!git pull -q
!pip install -q -r requirements.txt sentencepiece sacremoses

def check(p):
    f = os.path.join(p, "model.safetensors")
    print(p, "->", f"{os.path.getsize(f)/1e9:.2f} GB" if os.path.exists(f) else "MISSING")

# restore models from Drive (not the other way round)
os.makedirs("/content/models", exist_ok=True)
for src, dst in [("bge_m3_janora_v2", "bge_m3_janora"),
                 ("bge_reranker_janora_v2", "bge_reranker_janora")]:
    if not os.path.exists(f"/content/models/{dst}"):
        shutil.copytree(f"/content/drive/MyDrive/janora_models/{src}", f"/content/models/{dst}")
    check(f"/content/models/{dst}")

sys.path.insert(0, "/content/M.A.R.G")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/M.A.R.G
/content/models/bge_m3_janora -> 2.27 GB
/content/models/bge_reranker_janora -> 2.27 GB


In [8]:
import os
import sys

os.chdir("/content/M.A.R.G")
sys.path.insert(0, os.getcwd())

print("Current directory:", os.getcwd())
print("src exists:", os.path.isdir("src"))
print("src contents:", os.listdir("src"))

Current directory: /content/M.A.R.G
src exists: True
src contents: ['translation', 'nlu', 'reasoning', 'api', 'data_prep', '__init__.py', 'retrieval', 'qa', '__pycache__']


In [11]:
import sys
import os

PROJECT_ROOT = "/content/M.A.R.G"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Python path:", sys.path[0])

Project root: /content/M.A.R.G
Python path: /content/M.A.R.G


In [13]:
import os
import sys
import importlib.util

PROJECT_ROOT = "/content/M.A.R.G"

print("Current directory:", os.getcwd())
print("Project exists:", os.path.exists(PROJECT_ROOT))
print("src exists:", os.path.exists(os.path.join(PROJECT_ROOT, "src")))
print("src init exists:", os.path.exists(os.path.join(PROJECT_ROOT, "src", "__init__.py")))

sys.path.insert(0, PROJECT_ROOT)

print("\nPython executable:", sys.executable)
print("Python path:", sys.path)
print("Module spec:", importlib.util.find_spec("src"))

Current directory: /content/M.A.R.G
Project exists: True
src exists: True
src init exists: True

Python executable: /usr/bin/python3
Python path: ['/content/M.A.R.G', '/content/M.A.R.G', '/content/M.A.R.G', '/content/M.A.R.G', '/content/M.A.R.G', '/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython']
Module spec: None


In [31]:
import sys
sys.path.insert(0, "/content/M.A.R.G")

from src.data_prep import loaders as L
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker

docs = L.card_frame()

ret = HybridRetriever(
    docs,
    use_dense=True,
    model_name="/content/models/bge_m3_janora"
)

rr = CrossEncoderReranker(
    "/content/models/bge_reranker_janora"
)

def ask(q, k=3, pool=30):
    hits = ret.search(q, k=pool, mode="dense")
    order, _ = rr.rerank(q, hits["text"].tolist())

    print(f"\nQ: {q}")
    print("Retriever:", [r.name for r in hits.head(k).itertuples()])
    print("Reranked:", [r.name for r in hits.iloc[order[:k]].itertuples()])

ask("ഞാൻ ഹിമാചൽ പ്രദേശിൽ നിന്നുള്ള ഒരു എം.ടെക് (M.Tech) വിദ്യാർത്ഥിയാണ്. എനിക്ക് അപേക്ഷിക്കാവുന്ന സ്കോളർഷിപ്പുകൾ ഏതൊക്കെയാണെന്ന് നിർദ്ദേശിക്കാമോ?")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Q: ഞാൻ ഹിമാചൽ പ്രദേശിൽ നിന്നുള്ള ഒരു എം.ടെക് (M.Tech) വിദ്യാർത്ഥിയാണ്. എനിക്ക് അപേക്ഷിക്കാവുന്ന സ്കോളർഷിപ്പുകൾ ഏതൊക്കെയാണെന്ന് നിർദ്ദേശിക്കാമോ?
Retriever: ['Mukhiya Mantri Protsahan Yojna', 'Mukhya Mantri Vidyarthi Kalyan Yojana', 'Technical Education Scholarship Scheme For Diploma Level']
Reranked: ['Mukhiya Mantri Protsahan Yojna', 'Technical Education Scholarship Scheme For Diploma Level', 'Mukhya Mantri Vidyarthi Kalyan Yojana']


In [30]:
import sys; sys.path.insert(0, "/content/M.A.R.G")
from src.data_prep import loaders as L
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker
docs = L.card_frame()
ret = HybridRetriever(docs, use_dense=True, model_name="/content/models/bge_m3_janora")
rr  = CrossEncoderReranker("/content/models/bge_reranker_janora")

def ask(q, k=3, pool=30):
    hits = ret.search(q, k=pool, mode="dense")
    order, _ = rr.rerank(q, hits["text"].tolist())
    print(f"\nQ: {q}\n  retriever:", [r.name for r in hits.head(k).itertuples()])
    print("  reranked: ", [r.name for r in hits.iloc[order[:k]].itertuples()])

ask("I am a 45 year old farmer from Gujarat, income 80000, which scheme can help me?")
ask("mujhe scholarship chahiye main SC student hu")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Q: I am a 45 year old farmer from Gujarat, income 80000, which scheme can help me?
  retriever: ['National Mission for Sustainable Agriculture - Reinforced Area Development', 'Harvesting Tools Assistance for General Category (HRT-2)', 'National Mission on Edible Oil- Oil Palm: Drip Irrigation']
  reranked:  ['Wire Fencing Around the Farm for Crop Protection', 'Assistance for Area Expansion in Fruit crops', 'To Assist the Farmers for Construction of Community Base Water Tank/Storage Structure for Efficient Use of Water Through Drip Irrigation']

Q: mujhe scholarship chahiye main SC student hu
  retriever: ['Top Class Education For Scheduled Caste Students', 'Post-Graduate Scholarships for Professional Courses for SC/ST Candidates', 'Solar Energy Corporation of India (SECI) Internship']
  reranked:  ['Top Class Education For Scheduled Caste Students', 'Post-Graduate Scholarships for Professional Courses for SC/ST Candidates', 'Post Matric Scholarship for SC students (Delhi)']


In [29]:
import sys, importlib, os
os.chdir("/content/M.A.R.G")
if "/content/M.A.R.G" not in sys.path:
    sys.path.insert(0, "/content/M.A.R.G")
importlib.invalidate_caches()
for m in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[m]          # drop any stale/failed import

print("src/__init__.py:", os.path.exists("src/__init__.py"),
      "| data_prep:", os.path.isdir("src/data_prep"))
from src.data_prep import loaders as L
print("import OK")

src/__init__.py: True | data_prep: True
import OK


In [32]:
q = "ഞാൻ ഹിമാചൽ പ്രദേശിൽ നിന്നുള്ള ഒരു എം.ടെക് (M.Tech) വിദ്യാർത്ഥിയാണ്. എനിക്ക് അപേക്ഷിക്കാവുന്ന സ്കോളർഷിപ്പുകൾ ഏതൊക്കെയാണെന്ന് നിർദ്ദേശിക്കാമോ?"
ask(q)
ask("I am an M.Tech student from Himachal Pradesh. Which scholarships can I apply for?")   # English baseline

names = ["Mukhiya Mantri Protsahan Yojna", "Mukhya Mantri Vidyarthi Kalyan Yojana",
         "Technical Education Scholarship Scheme For Diploma Level"]
print(docs[docs.name.isin(names)].drop(columns=["text"], errors="ignore").T)


Q: ഞാൻ ഹിമാചൽ പ്രദേശിൽ നിന്നുള്ള ഒരു എം.ടെക് (M.Tech) വിദ്യാർത്ഥിയാണ്. എനിക്ക് അപേക്ഷിക്കാവുന്ന സ്കോളർഷിപ്പുകൾ ഏതൊക്കെയാണെന്ന് നിർദ്ദേശിക്കാമോ?
Retriever: ['Mukhiya Mantri Protsahan Yojna', 'Mukhya Mantri Vidyarthi Kalyan Yojana', 'Technical Education Scholarship Scheme For Diploma Level']
Reranked: ['Mukhiya Mantri Protsahan Yojna', 'Technical Education Scholarship Scheme For Diploma Level', 'Mukhya Mantri Vidyarthi Kalyan Yojana']

Q: I am an M.Tech student from Himachal Pradesh. Which scholarships can I apply for?
Retriever: ['Technical Education Scholarship Scheme For Diploma Level', 'Mukhiya Mantri Protsahan Yojna', 'Mukhya Mantri Vidyarthi Kalyan Yojana']
Reranked: ['Mukhiya Mantri Protsahan Yojna', 'Technical Education Scholarship Scheme For Diploma Level', 'Mukhya Mantri Vidyarthi Kalyan Yojana']
                                                       2689  \
slug                                                   mmpy   
name                         Mukhiya Mantri Protsahan Yoj